## Import ##

In [ ]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print('Device: ', device)

In [2]:
# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disable

## Functions ##

In [3]:
# Step 3: Function to Extract Embeddings for a List of Frames
def extract_video_embedding(image_list):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        for image in image_list:
            input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            features = model(input_tensor)
            embeddings.append(features.squeeze().cpu().numpy())
    return embeddings

## Process ##

In [4]:
from torch.utils.data import Dataset, DataLoader

class CustomImageDataset(Dataset):
    def __init__(self, video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256"):

        full_sample_folder_list = []
        labels = []
        process_count = 0
        for label_folder in os.listdir(video_folder):
            process_count += 1
            if process_count < 325:
                continue
            full_label_folder = os.path.join(video_folder, label_folder)
            label = int(label_folder)
            print("process_count: ", process_count, ' , label: ', label)
            for sample_folder in os.listdir(full_label_folder):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_list.append(full_sample_folder)
                labels.append(label)


        self.full_sample_folder_list = full_sample_folder_list
        self.labels = labels

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        input_tensor_list = []
        full_sample_folder = self.full_sample_folder_list[idx]
        for image_file in os.listdir(full_sample_folder):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            input_tensor_list.append(input_tensor)
        return input_tensor_list, self.labels[idx] 


In [5]:
batch_size = 1
test_dataset = CustomImageDataset()

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)  # Adjust batch size as needed


process_count:  325  , label:  94
process_count:  326  , label:  290
process_count:  327  , label:  738
process_count:  328  , label:  216
process_count:  329  , label:  227
process_count:  330  , label:  62
process_count:  331  , label:  531
process_count:  332  , label:  685
process_count:  333  , label:  44
process_count:  334  , label:  20
process_count:  335  , label:  494
process_count:  336  , label:  502
process_count:  337  , label:  27
process_count:  338  , label:  105
process_count:  339  , label:  588
process_count:  340  , label:  186
process_count:  341  , label:  620
process_count:  342  , label:  293
process_count:  343  , label:  693
process_count:  344  , label:  143
process_count:  345  , label:  627
process_count:  346  , label:  703
process_count:  347  , label:  426
process_count:  348  , label:  702
process_count:  349  , label:  599
process_count:  350  , label:  644
process_count:  351  , label:  578
process_count:  352  , label:  428
process_count:  353  , la

In [6]:
correct = 0
top_5_correct = 0
total = 0
running_loss = 0.0
# since we're not training, we don't need to calculate the gradients for our outputs
video_embeddings = []
video_labels = []
process_count = 0
with torch.no_grad():
    for image_list, label in test_loader:
        process_count += 1
        print("process_count: ", process_count, ' , label: ', label)
        video_embedding = extract_video_embedding(image_list)
        video_embeddings.append(video_embedding)
        video_labels.append(label)

with open('features_right_ha d_frames_large_loader.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

RuntimeError: Caught RuntimeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 302, in _worker_loop
    data = fetcher.fetch(index)
  File "/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 58, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 58, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/tmp/ipykernel_17556/3880576996.py", line 34, in __getitem__
    input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
  File "/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/torch/cuda/__init__.py", line 217, in _lazy_init
    raise RuntimeError(
RuntimeError: Cannot re-initialize CUDA in